In [130]:
!pip install unidecode

In [131]:
# explain all imports
import pandas as pd
import ast
import re
import unidecode

In [132]:
df = pd.read_csv('recipes.csv')
print(df)

       Unnamed: 0                                              Title  \
0               0  Miso-Butter Roast Chicken With Acorn Squash Pa...   
1               1                    Crispy Salt and Pepper Potatoes   
2               2                        Thanksgiving Mac and Cheese   
3               3                 Italian Sausage and Bread Stuffing   
4               4                                       Newton's Law   
...           ...                                                ...   
13496       13496                               Brownie Pudding Cake   
13497       13497  Israeli Couscous with Roasted Butternut Squash...   
13498       13498  Rice with Soy-Glazed Bonito Flakes and Sesame ...   
13499       13499                                        Spanakopita   
13500       13500  Mexican Poblano, Spinach, and Black Bean "Lasa...   

                                             Ingredients  \
0      ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...   
1      ['2 larg

In [133]:
df.columns

Index(['Unnamed: 0', 'Title', 'Ingredients', 'Instructions', 'Image_Name',
       'Cleaned_Ingredients'],
      dtype='object')

Read the datset and create a dataframw that just has the column we need (in this case the ingredients column).

In [134]:
df = df[["Ingredients"]]
print(df)

                                             Ingredients
0      ['1 (3½–4-lb.) whole chicken', '2¾ tsp. kosher...
1      ['2 large egg whites', '1 pound new potatoes (...
2      ['1 cup evaporated milk', '1 cup whole milk', ...
3      ['1 (¾- to 1-pound) round Italian loaf, cut in...
4      ['1 teaspoon dark brown sugar', '1 teaspoon ho...
...                                                  ...
13496  ['1 cup all-purpose flour', '2/3 cup unsweeten...
13497  ['1 preserved lemon', '1 1/2 pound butternut s...
13498  ['Leftover katsuo bushi (dried bonito flakes) ...
13499  ['1 stick (1/2 cup) plus 1 tablespoon unsalted...
13500  ['12 medium to large fresh poblano chiles (2 1...

[13501 rows x 1 columns]


Apply [ast.literal_eval](https://www.geeksforgeeks.org/python/difference-between-eval-and-ast-literal-eval-in-python/#overview-of-astliteral_eval), to ensure everything in the column is processed as basic literals like strings.

In [135]:
df["Ingredients"] = df["Ingredients"].apply(ast.literal_eval)

Try to catch common terms used in recipess well as certain puncuation, ranges and ces. I used the help of chatGPT (commentented) to achieve this,

In [136]:
REMOVE_WORDS = [
    "teaspoon", "teaspoons", "tsp", "tbsp", "tablespoon", "tablespoons",
    "cup", "cups", "ounce", "ounces", "oz", "pound", "pounds", "lb", "lbs",
    "gram", "grams", "g", "kilogram", "kg", "milliliter", "ml", "liter", "l",
    "quart", "large", "small", "medium", "mini", "fresh", "good-quality", "sturdy",
    "finely", "thinly", "melted", "divided", "plus", "more", "about",
    "room temperature", "torn", "cut", "sliced", "chopped", "cored", "salted",  "unsltated", 
    "garnish", "topping", "halved", "quatered", "whole", "kosher", "finely", "chopped",
    "drained", "roasted", "rinsed", "toasted", "cooked", "divided", "plus", "more",
    "and", "if", "canned", "leftover", "of", "stick", "to"
    "ground", "freshly", "pinch", "crushed", "pieces"
]

# ChatGPT helped me understand how to use this code and what I would need to code to get rid of as much of the unwanted terms as possible.
REMOVE_PATTERN = r"\b(" + "|".join([re.escape(w) for w in REMOVE_WORDS]) + r")\b"

def clean_ingredient(ing):
    ing = ing.lower()
    ing = unidecode.unidecode(ing)
    ing = re.sub(r"\d+[/\d\-]*", "", ing)
    ing = re.sub(r"[.,:;!?\"“”’'-]", " ", ing)
    ing = re.sub(r"\(.*?\)", "", ing)
    ing = re.sub(r"\s+", " ", ing)
    ing = re.sub(REMOVE_PATTERN, " ", ing)
    ing = re.sub(r"\d+[\/\d½¼¾–\-]*", "", ing)
    ing = re.sub(r",.*", "", ing)

    return ing.strip()

Apply cleaning of unwanted recipie terms, whitespace, and numbers.

In [137]:
df["cleaned_ingredients"] = df["Ingredients"].apply(
    lambda lst: [clean_ingredient(i) for i in lst]
)


In [139]:
print(df["cleaned_ingredients"])

0        [chicken, salt, acorn squash, sage, rosemary, ...
1        [large egg whites, new potatoes, salt, ground ...
2        [evaporated milk, milk, garlic powder, onion p...
3        [round italian loaf   into inch cubes, olive o...
4        [dark brown sugar, hot water, bourbon, lemon j...
                               ...                        
13496    [all purpose flour, unsweetened cocoa powder, ...
13497    [preserved lemon, butternut squash peeled   se...
13498    [katsuo bushi from making dashi or   katsuo bu...
13499    [unsalted butter, baby spinach, feta crumbled,...
13500    [to large   poblano chiles, can   tomatoes inc...
Name: cleaned_ingredients, Length: 13501, dtype: object


Turn each of the now cleaned ingredients into multiple rows with corresponding basket id's. I also drop any empty strings and duplicates per basket.

In [140]:
df["basket_id"] = df.index


In [141]:
baskets = df[["basket_id", "cleaned_ingredients"]].explode("cleaned_ingredients")


In [142]:
baskets = baskets.rename(columns={"cleaned_ingredients": "item_name"})


In [143]:
baskets = baskets[baskets["item_name"] != ""]


In [144]:
df = df.drop(columns=["Ingredients"])

In [145]:
baskets = baskets.drop_duplicates()
print(baskets)


       basket_id                                     item_name
0              0                                       chicken
0              0                                          salt
0              0                                  acorn squash
0              0                                          sage
0              0                                      rosemary
...          ...                                           ...
13500      13500                                       raisins
13500      13500                                corn tortillas
13500      13500                                   black beans
13500      13500                                     pine nuts
13500      13500  a to quart shallow flameproof casserole dish

[139317 rows x 2 columns]


Finish cleaning and export, ready to be used for task two (market basket analysis/ FP growth).

In [146]:
df_long = df.explode("cleaned_ingredients")
df_long = df_long.rename(columns={"cleaned_ingredients": "item_name"})


In [147]:
df_long = df_long[df_long["item_name"].notna()]
df_long = df_long[df_long["item_name"].str.strip() != ""]


In [148]:
df_long.head()


,item_name,basket_id
0,chicken,0
0,salt,0
0,acorn squash,0
0,sage,0
0,rosemary,0


In [149]:
df_long.to_csv("grocery_baskets.csv", index=False)
